In [1]:
%load_ext autoreload
%autoreload 2

# Lead Time Example
This is a basic example to test the lead time functionality.
The configuration is the following:
- One location ```location1``` and one commodity ```commodity1``` with unit ```unit1```
- Two investment periods ```0``` and ```1``` each 1 year
- One source ```source1``` producing ```commodity1``` with a constant maximum operation rate of 1
- One sink ```sink1``` consuming ```commodity1``` with a constant fixed operation rate of 0 in the first IP and 1 in the second IP

Expected behaviour without lead times:
- investment in 1 capacity of ```source1```in the second IP
- operation of ```source1``` in the second IP

Expected behaviour with a 1 year lead time for ```source1```:
- investment in 1 capacity of ```source1```in the first IP
- operation of ```source1``` in the second IP

Open questions:
- How do the costs differ?
    - Costs should be higher when considering lead times


In [2]:

import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np

In [3]:
esM = fn.EnergySystemModel(
    locations = {"location1", "location2"},
    commodities = {"commodity1"},
    commodityUnitsDict = {"commodity1": "unit1"},
    startYear = 0,
    numberOfInvestmentPeriods = 2,
    investmentPeriodInterval = 1
)

In [ ]:
esM.add(
    fn.Source(
        esM = esM,
        name = "source1",
        commodity = "commodity1",
        hasCapacityVariable = True,
        operationRateMax = {0: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
        capacityMax = {0: pd.Series({"location1": 100,
                                     "location2": 100}),
                       1: pd.Series({"location1": 100,
                                     "location2": 100})
                       },
        investPerCapacity = 1000,
        opexPerCapacity = 1020,
        interestRate = 0.08,
        economicLifetime = 1,
        leadTime = {0: pd.Series({"location1": 1, "location2": 1}),
                    1: pd.Series({"location1": 1, "location2": 0})
                    }
    )
)

C:\Users\j.becker\FINE\fine\component.py:785: UserWarning: Component identifier source1 already exists. Data will be overwritten.
  warnings.warn(


In [15]:
esM.add(
    fn.Sink(
        esM = esM,
        name = "sink1",
        commodity = "commodity1",
        hasCapacityVariable = False,
        operationRateFix = {0: pd.DataFrame(np.zeros((8760, 2)), columns=["location1", "location2"], index=range(8760)),
                            1: pd.DataFrame(np.ones((8760, 2)), columns=["location1", "location2"], index=range(8760))
                            },
    )
)

C:\Users\j.becker\FINE\fine\component.py:785: UserWarning: Component identifier sink1 already exists. Data will be overwritten.
  warnings.warn(


In [16]:
solver = fn.utils.ImplementedSolvers.STANDARD_SOLVER.value

# Code
esM.optimize(timeSeriesAggregation=False, solver='gurobi')

Set parameter OutputFlag to value 1
Set parameter Threads to value 3
Set parameter LogFile to value ""
Set parameter QCPDual to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 3 threads

Non-default parameters:
QCPDual  1
Threads  3

Optimize a model with 70092 rows, 70096 columns and 140184 nonzeros (Min)
Model fingerprint: 0xb192faee
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+03, 2e+03]
  Bounds range     [1e+00, 1e+02]
  RHS range        [0e+00, 0e+00]

Presolve removed 70092 rows and 70096 columns
Presolve time: 0.03s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.2000000e+03   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.05 secon

In [17]:
esM.getOptimizationSummary("SourceSinkModel", outputLevel=0, ip = 0)

location1 location2
Component Property                        Unit                            
sink1     NPVcontribution                 [1e9 Euro]         0.0       0.0
          TAC                             [1e9 Euro/a]       0.0       0.0
          capacity                        [unit1]            NaN       NaN
          capexCap                        [1e9 Euro/a]       NaN       NaN
          capexIfBuilt                    [1e9 Euro/a]       NaN       NaN
          commissioning                   [unit1]            NaN       NaN
          commodCosts                     [1e9 Euro/a]       0.0       0.0
          commodRevenues                  [1e9 Euro/a]       0.0       0.0
          decommissioning                 [unit1]            NaN       NaN
          invest                          [1e9 Euro]         NaN       NaN
          investLifetimeExtension         [1e9 Euro]         NaN       NaN
          isBuilt                         [-]                NaN       NaN
          operation                       [unit1*h]          0.0       0.0
          operation_annual                [unit1*h/a]        0.0       0.0
          opexCap                         [1e9 Euro/a]       NaN       NaN
          opexIfBuilt                     [1e9 Euro/a]       NaN       NaN
          opexOp                          [1e9 Euro/a]       0.0       0.0
          revenueLifetimeShorteningResale [1e9 Euro]         NaN       NaN
source1   NPVcontribution                 [1e9 Euro]      2100.0    2100.0
          TAC                             [1e9 Euro/a]    2100.0    2100.0
          capacity                        [unit1]            0.0       0.0
          capexCap                        [1e9 Euro/a]    1080.0    1080.0
          capexIfBuilt                    [1e9 Euro/a]       NaN       NaN
          commissioning                   [unit1]            1.0       1.0
          commodCosts                     [1e9 Euro/a]       0.0       0.0
          commodRevenues                  [1e9 Euro/a]       0.0       0.0
          decommissioning                 [unit1]            0.0       0.0
          invest                          [1e9 Euro]      1000.0    1000.0
          investLifetimeExtension         [1e9 Euro]           0         0
          isBuilt                         [-]                NaN       NaN
          operation                       [unit1*h]          0.0       0.0
          operation_annual                [unit1*h/a]        0.0       0.0
          opexCap                         [1e9 Euro/a]    1020.0    1020.0
          opexIfBuilt                     [1e9 Euro/a]       NaN       NaN
          opexOp                          [1e9 Euro/a]       0.0       0.0
          revenueLifetimeShorteningResale [1e9 Euro]           0         0